# 09_Gold_Patient

Gold-layer notebook. Attach `FHIR_Lakehouse` before running.

In [1]:
from pyspark.sql.functions import col, element_at, to_date, current_timestamp

spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

StatementMeta(, aa2bae64-a750-4d9e-8122-0616f19a14f8, 3, Finished, Available, Finished, False)

DataFrame[]

In [2]:
silver_patient_df = spark.table("silver.patient")
print("Silver Patient Records:", silver_patient_df.count())

StatementMeta(, aa2bae64-a750-4d9e-8122-0616f19a14f8, 4, Finished, Available, Finished, False)

Silver Patient Records: 1484


In [3]:
gold_patient_df = silver_patient_df.select(
    col("id").alias("patient_id"),
    element_at(col("name"), 1).getField("given")[0].alias("first_name"),
    element_at(col("name"), 1).getField("family").alias("last_name"),
    to_date(col("birthDate")).alias("birth_date"),
    col("gender"),
    col("active"),
    col("deceasedBoolean").alias("is_deceased"),
    element_at(col("address"), 1).getField("city").alias("city"),
    element_at(col("address"), 1).getField("state").alias("state"),
    element_at(col("address"), 1).getField("country").alias("country"),
    col("meta.lastUpdated").alias("source_last_updated"),
    col("extraction_timestamp"),
    col("load_timestamp")
).withColumn("gold_load_timestamp", current_timestamp())

StatementMeta(, aa2bae64-a750-4d9e-8122-0616f19a14f8, 5, Finished, Available, Finished, False)

In [4]:
(gold_patient_df.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true").saveAsTable("gold.dim_patient"))

StatementMeta(, aa2bae64-a750-4d9e-8122-0616f19a14f8, 6, Finished, Available, Finished, False)

In [5]:
spark.table("gold.dim_patient").printSchema()
spark.table("gold.dim_patient").show(10, truncate=False)
print("Total Gold Patient Records:", spark.table("gold.dim_patient").count())

StatementMeta(, aa2bae64-a750-4d9e-8122-0616f19a14f8, 7, Finished, Available, Finished, False)

root
 |-- patient_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- birth_date: date (nullable = true)
 |-- gender: string (nullable = true)
 |-- active: boolean (nullable = true)
 |-- is_deceased: boolean (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- source_last_updated: string (nullable = true)
 |-- extraction_timestamp: timestamp (nullable = true)
 |-- load_timestamp: timestamp (nullable = true)
 |-- gold_load_timestamp: timestamp (nullable = true)

+------------------------------------+----------+--------------+----------+-------+------+-----------+--------+------------+-------+-----------------------------+--------------------------+--------------------------+--------------------------+
|patient_id                          |first_name|last_name     |birth_date|gender |active|is_deceased|city    |state       |country|source_last